# 3D-VGA — step 1: text → speech

Synthesises one WAV per ScanRefer description with Piper: six voices, of which the last two are heard at validation only. Built for **Run All**; it resumes where it stopped.

Writes `data_audio/wav/<scene>/<object>_<ann>.wav` (16 kHz mono) and `data_audio/manifest_<split>.json`. Next: `run_audio_encode.ipynb`.

## 1 — configuration

The only cell you need to edit.

In [ ]:
# USER CONFIGURATION

DEVICE     = "auto"      # "auto" | "cpu" | "cuda"   -- auto probes for a working GPU
GPU_ID     = "0"
QUICK_TEST = False       # True: 2 scenes per split into data_audio/_quick -- minutes, and
                         #       never mixed with the real corpus

SPLITS = ["train", "val"]

RUN_TTS     = True
TTS_BACKEND = "piper"    # piper | coqui | espeak | gtts
VOICES = [               # piper voice ids; missing models are downloaded automatically
    "en_US-amy-medium", "en_US-ryan-medium", "en_GB-jenny_dioco-medium", "en_US-joe-medium",
    "en_US-lessac-medium", "en_GB-alan-medium",     # <- the last HELD_OUT_VOICES: val only
]
HELD_OUT_VOICES = 2      # the last N voices never speak a training utterance (speaker-OOD test)
TTS_ON_GPU      = "auto" # True | False | "auto" -- needs onnxruntime-gpu; falls back to CPU
TTS_INSTANCES   = 1      # CPU synthesis in N parallel processes (ignored on GPU)
OVERWRITE_WAVS  = False  # False resumes: WAVs that exist are kept

FRAME_CAP_SECONDS = 10.24 # what the encoding step keeps of each clip (256 frames at 25/s)

# GOOGLE DRIVE -- Colab only

MIRROR_TO_DRIVE = True
DRIVE_FOLDER    = "3DVGA"        # under MyDrive/
WAVS_ON_DRIVE   = True           # keep the WAVs on Drive, so a recycled runtime resumes

QUICK_SCENES = 2

## 2 — environment

Probes the GPU by launching a kernel — `torch.cuda.is_available()` alone can report `True` on an unsupported card.

In [ ]:
import glob as _glob
import importlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

REPO = Path.cwd()
RESULTS = {}

if not (REPO / "scripts" / "audio_train.py").is_file():
    raise SystemExit(f"Run this notebook from the repository root. Current cwd: {REPO}")

# lib/config.py lists this directory at import time; an empty one is enough for the steps
# that never read a scan.
(REPO / "data" / "scannet" / "scans").mkdir(parents=True, exist_ok=True)

IN_COLAB = importlib.util.find_spec("google.colab") is not None

GPU_USABLE, GPU_NAME, GPU_NOTE = False, None, ""
try:
    import torch
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        major, minor = torch.cuda.get_device_capability(0)
        try:
            _probe = torch.randn(8, 8, device="cuda")
            (_probe @ _probe).sum().item()
            GPU_USABLE = True
            GPU_NOTE = f"{GPU_NAME} (sm_{major}{minor})"
        except Exception as error:
            arch = getattr(torch.cuda, "get_arch_list", lambda: ["?"])()
            GPU_NOTE = (f"{GPU_NAME} (sm_{major}{minor}) present but NO KERNEL RUNS -- "
                        f"this torch ships {arch}. "
                        f"{type(error).__name__}: {str(error).splitlines()[0][:80]}")
    else:
        GPU_NOTE = "no CUDA device visible to torch"
except Exception as error:
    GPU_NOTE = f"torch unavailable: {error}"

if DEVICE == "auto":
    DEVICE = "cuda" if GPU_USABLE else "cpu"
elif DEVICE == "cuda" and not GPU_USABLE:
    print("[warn] DEVICE='cuda' but no kernel launches here.")

print("=" * 88)
print(f"environment  : {'Google Colab' if IN_COLAB else 'local machine'}")
print(f"python       : {sys.version.split()[0]}")
print(f"GPU          : {'USABLE -- ' + GPU_NOTE if GPU_USABLE else 'UNUSABLE -- ' + GPU_NOTE}")
print(f"device       : {DEVICE}")
print(f"quick test   : {QUICK_TEST}")
print("=" * 88)

### Runner

Streams a command's output and keeps its log under `outputs/reports/<step>/`.

In [ ]:
REPORTS = REPO / "outputs" / "reports"

def run(key, cmd, env=None, note=""):
    """Run one command, stream its output, and keep a report under outputs/reports/."""
    cmd = [str(c) for c in cmd]
    printable = " ".join(cmd)
    print("=" * 88)
    print(f"STEP   {key}" + (f"\n       {note}" if note else ""))
    print(f"$ {printable}\n{'-' * 88}")

    full_env = os.environ.copy()
    full_env.update({k: str(v) for k, v in (env or {}).items()})
    full_env["CUDA_VISIBLE_DEVICES"] = GPU_ID if (DEVICE == "cuda" and GPU_USABLE) else ""
    full_env["PYTHONUNBUFFERED"] = "1"

    started = time.time()
    captured = []
    process = subprocess.Popen(cmd, cwd=str(REPO), env=full_env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        captured.append(line)
    code = process.wait()
    elapsed = round(time.time() - started, 1)

    folder = REPORTS / key
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "log.txt").write_text("".join(captured))
    record = {"key": key, "note": note, "cmd": printable, "exit_code": code,
              "elapsed_sec": elapsed, "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
              "environment": "Google Colab" if IN_COLAB else "local", "gpu": GPU_NAME or "none"}
    (folder / "result.json").write_text(json.dumps(record, indent=2))
    RESULTS[key] = record

    if code in (247, -9, 137):
        verdict = f"FAILED (exit {code}) -- killed by the OS, out of host RAM"
    else:
        verdict = "OK" if code == 0 else f"FAILED (exit {code})"
    print(f"{'-' * 88}\n[{key}] {verdict} in {elapsed}s   log -> {folder.relative_to(REPO)}/log.txt")
    return code

## 3 — packages

Colab only. `onnxruntime-gpu` replaces the CPU build so Piper can synthesise on the GPU. If the GPU cannot run it, synthesis falls back to the CPU by itself.

In [ ]:
EXTRA_PACKAGES = ["piper-tts", "soundfile"]
REQUIRED_MODULES = ["torch", "numpy", "piper", "onnxruntime", "easydict"]

def _pip(*args):
    print("$ pip install " + " ".join(args))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

def ensure_packages(force=False):
    if not (IN_COLAB or force):
        print("not on Colab -- skipping installs. Pass force=True to run them anyway.")
        return
    if (REPO / "req" / "requirements.txt").is_file():
        _pip("-r", str(REPO / "req" / "requirements.txt"))
    _pip(*EXTRA_PACKAGES)

ensure_packages()

print("\n-- packages --")
MISSING_PACKAGES = []
for module in REQUIRED_MODULES:
    try:
        loaded = importlib.import_module(module)
        print(f"  {module:18s} OK    {getattr(loaded, '__version__', '')}")
    except Exception:
        MISSING_PACKAGES.append(module)
        print(f"  {module:18s} --    MISSING")
if MISSING_PACKAGES:
    print(f"\n  install first: pip install {' '.join(MISSING_PACKAGES)}")

if IN_COLAB and GPU_USABLE and TTS_ON_GPU and TTS_BACKEND == "piper":
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "onnxruntime"], check=False)
    _pip("onnxruntime-gpu")

TTS_GPU = False
if TTS_ON_GPU and GPU_USABLE and TTS_BACKEND == "piper":
    probe = subprocess.run(
        [sys.executable, "-c", "import onnxruntime as o; print(o.get_available_providers())"],
        capture_output=True, text=True)
    TTS_GPU = "CUDAExecutionProvider" in probe.stdout
print(f"\nspeech will be synthesised on: {'GPU' if TTS_GPU else 'CPU'}")

## 4 — Google Drive and paths

On Colab the WAVs are written straight to Drive. Locally everything stays under `data_audio/`.

In [ ]:
# Off Colab this whole cell is inert, so the notebook stays runnable locally.

DRIVE_ROOT = None

def mount_drive():
    """Mount Drive and return the destination directory, or None if unavailable."""
    global DRIVE_ROOT
    if not (MIRROR_TO_DRIVE and IN_COLAB):
        print("Drive off"
              + ("" if MIRROR_TO_DRIVE else " (MIRROR_TO_DRIVE=False)")
              + ("" if IN_COLAB else " -- not running on Colab, everything stays local"))
        return None
    try:
        from google.colab import drive
        mount = Path("/content/drive")
        if not (mount / "MyDrive").is_dir():
            drive.mount(str(mount))
        DRIVE_ROOT = mount / "MyDrive" / DRIVE_FOLDER
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print(f"Drive mounted -> {DRIVE_ROOT}")
    except Exception as error:
        # A failed mount must not take the run with it.
        DRIVE_ROOT = None
        print(f"Drive not mounted ({error}). The run continues; results stay on this "
              f"machine only and will be lost when the runtime recycles.")
    return DRIVE_ROOT

def copy_newer(source, target, quiet=False):
    """Copy a file or a tree, skipping files the target already has up to date."""
    source, target = Path(source), Path(target)
    if not source.exists():
        return 0
    files = [source] if source.is_file() else [p for p in source.rglob("*") if p.is_file()]
    copied = 0
    for path in files:
        if "tensorboard" in path.parts or path.suffix == ".tmp":
            continue
        dest = target if source.is_file() else target / path.relative_to(source)
        try:
            if dest.exists() and dest.stat().st_mtime >= path.stat().st_mtime \
                    and dest.stat().st_size == path.stat().st_size:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, dest)
            copied += 1
        except Exception as error:
            if not quiet:
                print(f"  could not copy {path.name}: {error}")
    return copied

def mirror_to_drive(quiet=False):
    """Copy outputs/ to Drive. Cheap to call repeatedly: only newer files are copied."""
    if DRIVE_ROOT is None:
        return 0
    copied = copy_newer(REPO / "outputs", DRIVE_ROOT / "outputs", quiet=quiet)
    if not quiet:
        print(f"mirrored {copied} new/changed file(s) to {DRIVE_ROOT}/outputs")
    return copied

def restore_from_drive(*relative):
    """Bring files back from Drive after a fresh runtime, where they are missing here."""
    if DRIVE_ROOT is None:
        return
    for rel in relative:
        copied = copy_newer(DRIVE_ROOT / rel, REPO / rel, quiet=True)
        if copied:
            print(f"restored {copied} file(s) from Drive: {rel}")

mount_drive()

LOCAL_ROOT = REPO / "data_audio" / ("_quick" if QUICK_TEST else "")
if DRIVE_ROOT is not None and WAVS_ON_DRIVE and not QUICK_TEST:
    AUDIO_ROOT = DRIVE_ROOT / "data_audio"
else:
    AUDIO_ROOT = LOCAL_ROOT
FEAT_DIR = LOCAL_ROOT / "feat"           # always local: training reads it at every step

DATA_ENV = {
    "VGA_AUDIO_ROOT": AUDIO_ROOT,
    "VGA_AUDIO_FEAT_DIR": FEAT_DIR,
    "VGA_HELD_OUT_VOICES": HELD_OUT_VOICES,
}
SUBSET = ["--num_scenes", QUICK_SCENES] if QUICK_TEST else []

if not QUICK_TEST:
    restore_from_drive("data_audio/manifest_train.json", "data_audio/manifest_val.json")

print(f"WAVs and manifests : {AUDIO_ROOT}")

## 5 — preflight

What is on disk and what is left to do. Changes nothing.

In [ ]:
def annotations(split):
    data = json.loads((REPO / "data" / f"ScanRefer_filtered_{split}.json").read_text())
    if QUICK_TEST:
        scenes = set(sorted({d["scene_id"] for d in data})[:QUICK_SCENES])
        data = [d for d in data if d["scene_id"] in scenes]
    return data

def manifest(split):
    path = AUDIO_ROOT / f"manifest_{split}.json"
    return json.loads(path.read_text()) if path.is_file() else None

def key_of(d):
    return f"{d['scene_id']}/{d['object_id']}/{d['ann_id']}"

print(f"{'split':<7} {'descriptions':>13} {'WAVs (manifest)':>16}")
print("-" * 40)
to_speak = 0
for split in SPLITS:
    wanted = {key_of(d) for d in annotations(split)}
    listed = manifest(split)
    spoken = {e["key"] for e in (listed or {}).get("entries", []) if "error" not in e} & wanted
    to_speak += len(wanted) - len(spoken)
    print(f"{split:<7} {len(wanted):>13} {len(spoken):>16}")

print(f"\n{to_speak} clip(s) to synthesise.")
if to_speak and not TTS_GPU:
    print("CPU synthesis takes roughly 0.3-1 s per clip depending on the cores available; "
          "the full corpus is hours.\nIt resumes, so an interrupted session loses nothing.")

## 6 — synthesise

The voice is a fixed function of the annotation, so re-running assigns the same voices. With `TTS_INSTANCES > 1` several CPU processes share the work.

In [ ]:
# switch: RUN_TTS
if RUN_TTS:
    cmd = [sys.executable, "scripts/audio_tts/build_audioset.py", "--splits", *SPLITS,
           "--backend", TTS_BACKEND, *SUBSET]
    cmd = [str(c) for c in cmd]
    if TTS_BACKEND == "piper":
        cmd += ["--voice_list", *VOICES]
    if OVERWRITE_WAVS:
        cmd += ["--overwrite"]

    if TTS_GPU:
        cmd += ["--cuda"]
    elif TTS_INSTANCES > 1:
        # Each instance takes every N-th description and keeps its own manifest; the
        # ordinary run below then finds every WAV in place and merges the manifests.
        env = os.environ.copy()
        env.update({k: str(v) for k, v in DATA_ENV.items()})
        (REPORTS / "audio-tts").mkdir(parents=True, exist_ok=True)
        workers = []
        for i in range(1, TTS_INSTANCES + 1):
            log = open(REPORTS / "audio-tts" / f"instance_{i}.log", "w")
            workers.append(subprocess.Popen([*cmd, "--shard", f"{i}/{TTS_INSTANCES}"],
                                            cwd=str(REPO), env=env, stdout=log,
                                            stderr=subprocess.STDOUT))
        print(f"{TTS_INSTANCES} instances running; logs in outputs/reports/audio-tts/")
        codes = [w.wait() for w in workers]
        print("instance exit codes:", codes)

    run("audio-tts", cmd, env=DATA_ENV, note="ScanRefer descriptions -> WAVs")
    # Later notebooks read the manifests from the local tree.
    for split in SPLITS:
        copy_newer(AUDIO_ROOT / f"manifest_{split}.json", LOCAL_ROOT / f"manifest_{split}.json")
else:
    print("RUN_TTS is off")

## 7 — listen

Three random clips with their text and voice, clip lengths, and how many clips each voice spoke.

In [ ]:
import random
import numpy as np
from IPython.display import Audio, display

listed = manifest("val") or manifest(SPLITS[0])
if listed is None:
    print("no manifest yet -- run stage 1")
else:
    spoken = [e for e in listed["entries"] if "duration" in e]
    for entry in random.Random(0).sample(spoken, min(3, len(spoken))):
        scene, obj, ann = entry["key"].split("/")
        print(f"\n{entry['key']}   voice={entry['voice']}   {entry['duration']} s\n  {entry['text']}")
        display(Audio(str(AUDIO_ROOT / "wav" / scene / f"{obj}_{ann}.wav")))

    durations = np.array([e["duration"] for e in spoken])
    cap = FRAME_CAP_SECONDS
    print(f"\n{len(spoken)} clip(s), {len(listed['voices'])} voice(s)")
    print(f"duration  mean {durations.mean():.2f} s   median {np.median(durations):.2f} s   "
          f"p95 {np.percentile(durations, 95):.2f} s")
    print(f"{100 * (durations > cap).mean():.1f}% are longer than the {cap:.2f} s cap "
          f"the encoding step keeps, and will be centre-cropped")
    voices = {}
    for entry in spoken:
        voices[entry["voice"]] = voices.get(entry["voice"], 0) + 1
    print("clips per voice:", voices)
    print("\nNext: run_audio_encode.ipynb")